# Landmark v3: regularized BiGRU validation experiment

This notebook tries a small regularization package on the already cached
LRW-pretrained features. It uses only train and validation. The frozen final
test is not loaded and must not be used to select this model.

Compared configurations:

1. `feature_aug_ls`: temporal masking, small feature noise, feature dropout,
   label smoothing, and stronger classifier dropout/weight decay.
2. `feature_aug_ls_layernorm`: the same package plus input LayerNorm.


## 1. Mount Drive and load cached landmark-v3 features

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from collections import Counter
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
baseline_dir = drive_root / 'controlled_lrw_frontend_bigru_landmark_v3'
output_dir = drive_root / 'controlled_lrw_landmark_v3_regularized_bigru'
output_dir.mkdir(parents=True, exist_ok=True)

train_cache_path = baseline_dir / 'landmark_v3_train_lrw_resnet18_features.pt'
val_cache_path = baseline_dir / 'landmark_v3_val_lrw_resnet18_features.pt'
baseline_runs_path = baseline_dir / 'lrw_frontend_bigru_landmark_v3_multiseed_runs.csv'
reference_checkpoint = baseline_dir / 'lrw_frontend_bigru_landmark_v3_seed42_best.pt'

for required_path in [
    train_cache_path, val_cache_path, baseline_runs_path, reference_checkpoint
]:
    assert required_path.exists(), f'Missing required artifact: {required_path}'

train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)
vocab = load_torch_file(reference_checkpoint)['vocab']
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}
baseline_runs = pd.read_csv(baseline_runs_path)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

assert train_cache['features'].shape == (1085, 24, 512)
assert val_cache['features'].shape == (105, 24, 512)
assert set(train_cache['speakers']).isdisjoint(set(val_cache['speakers']))

feature_scale = float(train_cache['features'].float().std())
print('device:', device)
print('train features:', train_cache['features'].shape)
print('validation features:', val_cache['features'].shape)
print('feature scale:', feature_scale)
print('baseline mean macro-F1:', baseline_runs['validation_macro_f1'].mean())
print('FINAL TEST IS NOT LOADED.')


## 2. Dataset, model, metrics, and balanced sampler

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class FeatureSequenceDataset(Dataset):
    def __init__(
        self,
        cache,
        augment=False,
        max_temporal_mask=0,
        noise_fraction=0.0,
        feature_dropout=0.0,
    ):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )
        self.augment = augment
        self.max_temporal_mask = max_temporal_mask
        self.noise_fraction = noise_fraction
        self.feature_dropout = feature_dropout

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        features = self.features[index].clone()
        if self.augment:
            if self.noise_fraction > 0:
                features += torch.randn_like(features) * (
                    self.noise_fraction * feature_scale
                )
            if self.feature_dropout > 0:
                features = F.dropout(
                    features, p=self.feature_dropout, training=True
                )
            if self.max_temporal_mask > 0:
                mask_length = random.randint(0, self.max_temporal_mask)
                if mask_length:
                    start = random.randint(0, features.shape[0] - mask_length)
                    features[start:start + mask_length] = 0
        return features, self.labels[index]


class RegularizedBiGRU(nn.Module):
    def __init__(
        self,
        num_classes,
        input_dim=512,
        hidden_dim=128,
        dropout=0.55,
        use_layer_norm=False,
    ):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim) if use_layer_norm else nn.Identity()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        features = self.input_norm(features)
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def make_loaders(seed, config):
    group_keys = list(zip(train_cache['speakers'], train_cache['words']))
    group_counts = Counter(group_keys)
    weights = torch.tensor(
        [1.0 / group_counts[key] for key in group_keys], dtype=torch.double
    )
    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    train_dataset = FeatureSequenceDataset(
        train_cache,
        augment=True,
        max_temporal_mask=config['max_temporal_mask'],
        noise_fraction=config['noise_fraction'],
        feature_dropout=config['feature_dropout'],
    )
    val_dataset = FeatureSequenceDataset(val_cache, augment=False)
    return (
        DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=0),
        DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0),
        len(group_counts),
    )


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    class_rows = []
    for class_id in range(num_classes):
        tp = int(matrix[class_id, class_id])
        support = int(matrix[class_id].sum())
        predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'word': idx_to_word[class_id],
            'support': support,
            'predicted_count': predicted,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active = [row for row in class_rows if row['support']]
    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean([row['f1'] for row in active])),
        'balanced_accuracy': float(np.mean([row['recall'] for row in active])),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            outputs = model(features)
            total_loss += criterion(outputs, targets).item() * targets.size(0)
            total += targets.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(targets.cpu().tolist())
    metrics = classification_metrics(preds, labels, len(vocab))
    metrics['loss'] = total_loss / total
    metrics['preds'] = preds
    metrics['labels'] = labels
    return metrics


## 3. Controlled regularization configurations

In [ ]:
CONFIGS = {
    'feature_aug_ls': {
        'use_layer_norm': False,
        'dropout': 0.55,
        'max_temporal_mask': 4,
        'noise_fraction': 0.02,
        'feature_dropout': 0.05,
        'label_smoothing': 0.10,
        'weight_decay': 1e-3,
        'learning_rate': 5e-4,
    },
    'feature_aug_ls_layernorm': {
        'use_layer_norm': True,
        'dropout': 0.55,
        'max_temporal_mask': 4,
        'noise_fraction': 0.02,
        'feature_dropout': 0.05,
        'label_smoothing': 0.10,
        'weight_decay': 1e-3,
        'learning_rate': 5e-4,
    },
}

SEEDS = [42, 123, 2026]
print(pd.DataFrame(CONFIGS).T)
print('baseline mean validation macro-F1:', baseline_runs['validation_macro_f1'].mean())


In [ ]:
def run_one_seed(variant, config, seed, num_epochs=50, patience=10):
    set_reproducible_seed(seed)
    train_loader, val_loader, group_count = make_loaders(seed, config)
    set_reproducible_seed(seed)

    model = RegularizedBiGRU(
        len(vocab),
        dropout=config['dropout'],
        use_layer_norm=config['use_layer_norm'],
    ).to(device)
    criterion = nn.CrossEntropyLoss(
        label_smoothing=config['label_smoothing']
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )

    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    best_row = None
    best_state = None
    best_metrics = None
    epochs_without_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss_sum += loss.item() * targets.size(0)
            train_correct += (outputs.argmax(dim=1) == targets).sum().item()
            train_total += targets.size(0)

        val_metrics = evaluate(model, val_loader, criterion)
        train_loss = train_loss_sum / train_total
        train_accuracy = train_correct / train_total
        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )

        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            epochs_without_improvement = 0
            best_row = {
                'variant': variant,
                'seed': seed,
                'best_epoch': epoch,
                'train_accuracy_at_best': train_accuracy,
                'train_loss_at_best': train_loss,
                'validation_loss': val_metrics['loss'],
                'validation_accuracy': val_metrics['accuracy'],
                'validation_macro_f1': val_metrics['macro_f1'],
                'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
                'predicted_classes': val_metrics['predicted_classes'],
                'speaker_word_groups': group_count,
                'trainable_parameters': count_parameters(model),
            }
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            best_metrics = val_metrics
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    best_row['epochs_completed'] = epoch
    best_row['training_seconds'] = time.perf_counter() - start
    torch.save({
        'model_state_dict': best_state,
        'vocab': vocab,
        'config': config,
        'result': best_row,
        'validation_preds': best_metrics['preds'],
        'validation_labels': best_metrics['labels'],
        'test_used': False,
    }, output_dir / f'{variant}_seed{seed}_best.pt')
    pd.DataFrame(best_metrics['class_rows']).to_csv(
        output_dir / f'{variant}_seed{seed}_per_class.csv', index=False
    )

    print(
        f"{variant} seed={seed} | epoch={best_row['best_epoch']} | "
        f"train_acc={best_row['train_accuracy_at_best']:.4f} | "
        f"val_acc={best_row['validation_accuracy']:.4f} | "
        f"macro-F1={best_row['validation_macro_f1']:.4f} | "
        f"balanced_acc={best_row['validation_balanced_accuracy']:.4f} | "
        f"classes={best_row['predicted_classes']}/10"
    )
    return best_row


## 4. Run three seeds and compare with the frozen baseline

In [ ]:
rows = []
for variant, config in CONFIGS.items():
    for seed in SEEDS:
        rows.append(run_one_seed(variant, config, seed))

regularized_runs = pd.DataFrame(rows).sort_values(['variant', 'seed'])
regularized_runs.to_csv(
    output_dir / 'regularized_bigru_multiseed_runs.csv', index=False
)
display(regularized_runs)
print('FINAL TEST WAS NOT USED.')


In [ ]:
summary = regularized_runs.groupby('variant').agg(
    validation_accuracy_mean=('validation_accuracy', 'mean'),
    validation_accuracy_std=('validation_accuracy', 'std'),
    validation_macro_f1_mean=('validation_macro_f1', 'mean'),
    validation_macro_f1_std=('validation_macro_f1', 'std'),
    validation_balanced_accuracy_mean=('validation_balanced_accuracy', 'mean'),
    validation_balanced_accuracy_std=('validation_balanced_accuracy', 'std'),
    train_accuracy_mean=('train_accuracy_at_best', 'mean'),
).reset_index()

baseline_row = pd.DataFrame([{
    'variant': 'frozen_baseline',
    'validation_accuracy_mean': baseline_runs['validation_accuracy'].mean(),
    'validation_accuracy_std': baseline_runs['validation_accuracy'].std(),
    'validation_macro_f1_mean': baseline_runs['validation_macro_f1'].mean(),
    'validation_macro_f1_std': baseline_runs['validation_macro_f1'].std(),
    'validation_balanced_accuracy_mean': baseline_runs['validation_balanced_accuracy'].mean(),
    'validation_balanced_accuracy_std': baseline_runs['validation_balanced_accuracy'].std(),
    'train_accuracy_mean': baseline_runs['train_accuracy_at_best'].mean(),
}])

comparison = pd.concat([baseline_row, summary], ignore_index=True)
comparison['macro_f1_delta_vs_baseline'] = (
    comparison['validation_macro_f1_mean']
    - baseline_runs['validation_macro_f1'].mean()
)
comparison.to_csv(
    output_dir / 'regularized_bigru_vs_baseline_summary.csv', index=False
)
display(comparison.sort_values('validation_macro_f1_mean', ascending=False))

best_new = summary.sort_values(
    'validation_macro_f1_mean', ascending=False
).iloc[0]
delta = (
    best_new['validation_macro_f1_mean']
    - baseline_runs['validation_macro_f1'].mean()
)
print('best new variant:', best_new['variant'])
print('macro-F1 delta vs baseline:', delta)
if delta >= 0.01:
    print('PASS: regularization improved validation macro-F1 by at least 0.01.')
else:
    print('NO CLEAR GAIN: prioritize more train speakers or encoder fine-tuning.')
print('Do not evaluate these variants on the frozen final test.')
